***1) PREPROCESSING AND FILTERING***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "01D_GSE135779_ADULT_PREPROCESSING")


In [ ]:
#CREATE INDIVIDUAL ADATA FILES

import os
import glob
import gzip
import pandas as pd
import scanpy as sc
from scipy.io import mmread

# -------------------------------
# paths
# -------------------------------
raw_dir = f"{BASE_DIR}/GSE135779_RAW"
out_dir = f"{BASE_DIR}/adult_individual_h5ad"
os.makedirs(out_dir, exist_ok=True)

# -------------------------------
# adult GSM IDs
# -------------------------------
adult_gsms = [
    "GSM4029950", "GSM4029951", "GSM4029952", "GSM4029943", "GSM4029944",
    "GSM4029945", "GSM4029946", "GSM4029940", "GSM4029942", "GSM4029947",
    "GSM4029948", "GSM4029949"
]

# -------------------------------
# helper functions
# -------------------------------
def find_one_file(patterns):
    """Return first existing match across a list of glob patterns."""
    for pattern in patterns:
        hits = glob.glob(pattern)
        if hits:
            return hits[0]
    return None

def read_table_auto(path, header=None):
    """Read .tsv or .tsv.gz."""
    return pd.read_csv(path, sep="\t", header=header)

def make_unique(names):
    """Simple replacement for older pandas ParserBase dedup."""
    seen = {}
    out = []
    for x in names:
        x = "NA" if pd.isna(x) or str(x).strip() == "" else str(x)
        if x not in seen:
            seen[x] = 0
            out.append(x)
        else:
            seen[x] += 1
            out.append(f"{x}-{seen[x]}")
    return pd.Index(out)

# -------------------------------
# load shared genes file
# -------------------------------
genes_path = os.path.join(raw_dir, "GSE135779_genes.tsv.gz")
genes = read_table_auto(genes_path, header=None)

print("Genes file shape:", genes.shape)

# Common formats:
# col0 = Ensembl/gene_id
# col1 = gene_symbol
if genes.shape[1] >= 2:
    gene_ids = genes.iloc[:, 0].astype(str).values
    gene_symbols = genes.iloc[:, 1].astype(str).values
else:
    gene_ids = genes.iloc[:, 0].astype(str).values
    gene_symbols = genes.iloc[:, 0].astype(str).values

gene_symbols = make_unique(gene_symbols)

# -------------------------------
# create one h5ad per adult GSM
# -------------------------------
made_files = []
missing = []

for gsm in adult_gsms:
    # find matrix and barcode files for this GSM
    matrix_path = find_one_file([
        os.path.join(raw_dir, f"{gsm}_*_matrix.mtx.gz"),
        os.path.join(raw_dir, f"{gsm}_*_matrix.mtx")
    ])

    barcodes_path = find_one_file([
        os.path.join(raw_dir, f"{gsm}_*_barcodes.tsv.gz"),
        os.path.join(raw_dir, f"{gsm}_*_barcodes.tsv")
    ])

    if matrix_path is None or barcodes_path is None:
        missing.append((gsm, matrix_path, barcodes_path))
        print(f"Skipping {gsm}: missing file(s)")
        continue

    # sample name = filename stem before "_matrix"
    # example: GSM4029907_JB17010
    base = os.path.basename(matrix_path)
    sample_name = base.replace("_matrix.mtx.gz", "").replace("_matrix.mtx", "")

    print(f"Processing {sample_name}...")

    # read sparse matrix
    X = mmread(matrix_path).tocsr()

    # GEO matrices are usually genes x cells -> transpose to cells x genes
    # we will confirm against barcode and gene lengths
    barcodes = read_table_auto(barcodes_path, header=None).iloc[:, 0].astype(str).values

    if X.shape[0] == len(gene_symbols) and X.shape[1] == len(barcodes):
        X = X.T   # convert to cells x genes
    elif X.shape[0] == len(barcodes) and X.shape[1] == len(gene_symbols):
        pass      # already cells x genes
    else:
        raise ValueError(
            f"{sample_name}: matrix shape {X.shape} does not match "
            f"{len(gene_symbols)} genes and {len(barcodes)} barcodes"
        )

    # build AnnData
    adata = sc.AnnData(X=X)

    # unique cell names: sample_barcode
    adata.obs_names = [f"{sample_name}_{bc}" for bc in barcodes]
    adata.var_names = gene_symbols

    # store extra gene/sample metadata
    adata.var["gene_ids"] = gene_ids
    adata.var["gene_symbols"] = adata.var_names
    adata.obs["gsm_id"] = gsm
    adata.obs["sample"] = sample_name

    # save
    out_path = os.path.join(out_dir, f"{sample_name}.h5ad")
    adata.write_h5ad(out_path)
    made_files.append(out_path)

print("\nDone.")
print(f"Created {len(made_files)} h5ad files.")
if missing:
    print("\nMissing files for these GSMs:")
    for item in missing:
        print(item)

**Per-sample QC filtering.** Cells are filtered to `min_genes=200`, genes to `min_cells=3`, and a mitochondrial-fraction cutoff of `pct_counts_mt < 25` is applied per sample before combining. This mirrors the child pipeline (1A) exactly, for consistency across cohorts. 25% is a deliberately permissive ("very mild") threshold rather than the more common 10-20% used in cleaner PBMC datasets, chosen because this cohort spans many separately-processed patient samples with variable background mitochondrial content; a stricter global cutoff risked disproportionately removing otherwise-healthy cells from a subset of samples.

In [ ]:
# Sensitivity of cell retention to reasonable QC threshold choices.
from glob import glob
from qc_utils import export_qc_threshold_sensitivity

adult_qc_sensitivity = export_qc_threshold_sensitivity(
    glob(f"{BASE_DIR}/adult_individual_h5ad/GSM*.h5ad"),
    f"{BASE_DIR}/Results/qc/adult_qc_threshold_sensitivity.csv",
)
display(adult_qc_sensitivity.head())


In [ ]:
#FILTER CELLS AND GENES TO MAKE INDIVIDUAL FILES BEFORE COMBINING -- RAM

import os
import glob
import scanpy as sc
from analysis_config import MIN_GENES_PER_CELL, MAX_MITOCHONDRIAL_PERCENT

in_dir = f"{BASE_DIR}/adult_individual_h5ad"
out_dir = f"{BASE_DIR}/adult_individual_h5ad_filtered"
os.makedirs(out_dir, exist_ok=True)

h5ad_files = sorted(glob.glob(os.path.join(in_dir, "GSM*.h5ad")))
print(f"Found {len(h5ad_files)} files")

for f in h5ad_files:
    sample_name = os.path.basename(f)
    print(f"\nProcessing {sample_name}")

    adata = sc.read_h5ad(f)
    print("  original shape:", adata.shape)

    # basic per-cell QC metric
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

    # filter cells and genes
    sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
    sc.pp.filter_genes(adata, min_cells=3)

    # very mild mitochondrial filtering
    adata = adata[adata.obs["pct_counts_mt"] < MAX_MITOCHONDRIAL_PERCENT].copy()

    print("  filtered shape:", adata.shape)

    out_path = os.path.join(out_dir, sample_name)
    adata.write_h5ad(out_path)
    print("  saved:", out_path)

In [ ]:
# SCRUBLET — BATCH DOUBLET FILTERING PER GSM FILE

import scanpy as sc
import scrublet as scr
from analysis_config import EXPECTED_DOUBLET_RATE
import numpy as np
from scipy import sparse
import os

# ===== PATHS =====
input_dir = f"{BASE_DIR}/adult_individual_h5ad_filtered"
output_dir = f"{BASE_DIR}/adult_individual_nodoublet_h5ad"

os.makedirs(output_dir, exist_ok=True)
doublet_qc_records = []

# ===== LOOP THROUGH FILES =====
for file in os.listdir(input_dir):

    if file.startswith("GSM") and file.endswith(".h5ad"):

        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, file)

        print("\n==============================")
        print(f"Processing: {file}")

        # ===== LOAD =====
        adata = sc.read_h5ad(input_path)
        print("Original shape:", adata.shape)

        # ===== CHECK RAW COUNTS =====
        if np.issubdtype(adata.X.dtype, np.floating):
            print("⚠️ WARNING: Data may already be normalized/logged")

        # ===== PREP MATRIX =====
        counts_matrix = adata.X
        if sparse.issparse(counts_matrix):
            counts_matrix = counts_matrix.tocsc()
        else:
            counts_matrix = np.array(counts_matrix)

        # ===== RUN SCRUBLET =====
        scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=EXPECTED_DOUBLET_RATE, random_state=0)

        doublet_scores, predicted_doublets = scrub.scrub_doublets(
            min_counts=2,
            min_cells=3,
            min_gene_variability_pctl=85,
            n_prin_comps=30
        )

        # ===== STORE =====
        adata.obs["doublet_score"] = doublet_scores
        adata.obs["predicted_doublet"] = predicted_doublets.astype(bool)

        print("Doublet counts:")
        print(adata.obs["predicted_doublet"].value_counts())
        print("Doublet fraction:", adata.obs["predicted_doublet"].mean())

        # ===== FILTER =====
        adata_clean = adata[~adata.obs["predicted_doublet"]].copy()
        print("After removal:", adata_clean.shape)
        doublet_qc_records.append({
            "sample_file": file, "n_cells_before_doublet_filter": adata.n_obs,
            "n_predicted_doublets": int(predicted_doublets.sum()),
            "predicted_doublet_fraction": float(predicted_doublets.mean()),
            "n_cells_after_doublet_filter": adata_clean.n_obs,
        })

        # ===== SAVE =====
        adata_clean.write(output_path)
        print(f"Saved to: {output_path}")

from pathlib import Path
import pandas as pd
doublet_qc_path = Path(BASE_DIR) / "Results" / "qc" / "adult_doublet_qc.csv"
doublet_qc_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(doublet_qc_records).to_csv(doublet_qc_path, index=False)

print("\n✅ DONE: All GSM files processed")

**Gene universe handling.** Unlike the child pipeline (which uses an inner join across per-sample-filtered genes), the adult combine step below uses `join="outer"`, keeping the union of all genes seen across samples (missing values filled as needed by AnnData). This avoids the order-dependent gene-set shrinkage the child pipeline exhibits, at the cost of a sparser gene matrix for samples that didn't detect a given gene.

In [ ]:
#COMBINE H5AD FILES INTO SINGLE ONE

import os
import glob
import scanpy as sc
import anndata as ad

# -------------------------------
# paths
# -------------------------------
in_dir = f"{BASE_DIR}/adult_individual_nodoublet_h5ad"
out_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_combined_filtered.h5ad"

# -------------------------------
# find all adult h5ad files
# -------------------------------
h5ad_files = sorted(glob.glob(os.path.join(in_dir, "*.h5ad")))

print(f"Found {len(h5ad_files)} adult h5ad files")
for f in h5ad_files[:5]:
    print(" ", os.path.basename(f))

# -------------------------------
# read them in
# -------------------------------
adatas = []
for f in h5ad_files:
    adata = sc.read_h5ad(f)

    # make sure sample column exists
    if "sample" not in adata.obs.columns:
        sample_name = os.path.basename(f).replace(".h5ad", "")
        adata.obs["sample"] = sample_name

    adatas.append(adata)

print(f"Loaded {len(adatas)} AnnData objects")

# -------------------------------
# combine
# -------------------------------
adata_adult = ad.concat(
    adatas,
    axis=0,                # stack cells
    join="inner",          # shared gene universe, matching pediatric preprocessing
    merge="same",
    label="batch_file",
    keys=[os.path.basename(f).replace(".h5ad", "") for f in h5ad_files],
    index_unique=None
)

# -------------------------------
# cleanup
# -------------------------------
# make obs_names unique just in case
adata_adult.obs_names_make_unique()

print(adata_adult)
print(adata_adult.obs.head())
print(adata_adult.var.head())

# -------------------------------
# save
# -------------------------------
adata_adult.write_h5ad(out_path)
print(f"Saved filtered, combined adult AnnData to:\n{out_path}")

In [ ]:
import scanpy as sc
from analysis_config import N_HIGHLY_VARIABLE_GENES

# load filtered object
adata = sc.read_h5ad(
    f"{BASE_DIR}/adult_individual_h5ad/adata_adult_combined_filtered.h5ad"
)

print("Loaded:", adata)

# keep raw counts before normalization
adata.layers["counts"] = adata.X.copy()

# normalize and log transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# save log-normalized matrix in .raw
adata.raw = adata

# highly variable genes
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HIGHLY_VARIABLE_GENES,
    flavor="seurat"
)

print("Done")
print(adata)
print("Highly variable genes:", int(adata.var["highly_variable"].sum()))

out_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_combined_filtered2.h5ad"
adata.write_h5ad(out_path)

**Clustering resolution and HVG selection.** Five Leiden resolutions (0.2, 0.4, 0.6, 0.8, 1.0) are computed for comparison; **0.8** was selected as the final resolution used downstream (1E), matching the child pipeline's choice, because it was the finest resolution that still produced clusters mapping cleanly onto recognizable PBMC cell types via marker genes without fragmenting any single cell type across multiple clusters.

Note also that highly-variable-gene selection (previous step) is **not** batch-aware (no `batch_key="sample"`), so sample-to-sample technical variation could in principle be selected into the HVG set ahead of true biological variation, before Harmony ever runs. Harmony's batch correction on the resulting PCA embedding substantially mitigates this in practice (see the before/after UMAP-by-sample comparison in `Notebook_Outputs/batch_check_adult_umap_by_sample*.png`), but a batch-aware HVG selection would be a more rigorous alternative if this pipeline is revisited.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(
    f"{BASE_DIR}/adult_individual_h5ad/adata_adult_combined_filtered2.h5ad"
)

print("Loaded:", adata)
print("HVGs:", int(adata.var["highly_variable"].sum()))

# PCA directly on HVGs, no scaling
sc.tl.pca(adata, svd_solver="arpack", use_highly_variable=True, random_state=0)

# Batch correction across donors/samples (Harmony).
# harmonypy 0.0.10 provides an OS-independent wheel; Scanpy's wrapper handles
# that release's corrected-PC orientation and stores cells x components.
sc.external.pp.harmony_integrate(
    adata,
    key="sample",
    basis="X_pca",
    adjusted_basis="X_pca_harmony",
    random_state=0,
)
if adata.obsm["X_pca_harmony"].shape != adata.obsm["X_pca"].shape:
    raise ValueError("Harmony returned an unexpected corrected-PC shape.")

# neighbors + UMAP (on the batch-corrected embedding)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_harmony", random_state=0)
sc.tl.umap(adata, random_state=0)

# Leiden at multiple resolutions
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0]
for res in resolutions:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, random_state=0)
    print(key, "->", adata.obs[key].nunique(), "clusters")

# save
out_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_processed_multi_leiden.h5ad"
adata.write_h5ad(out_path)
print("Saved:", out_path)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.2", "leiden_res_0.4", "leiden_res_0.6", "leiden_res_0.8", "leiden_res_1.0"],
    wspace=0.4
, save="_figure_01.png")

In [ ]:
adata.write_h5ad(f"{BASE_DIR}/adult_individual_h5ad/adata_adult_processed_selected_leiden.h5ad")

In [ ]:
# Publication QC summary from the retained post-filter object.
from qc_utils import export_sample_qc

qc_table = export_sample_qc(adata, f"{BASE_DIR}/Results/qc/adult_postfilter_sample_qc.csv")
display(qc_table)


In [ ]:
# Remove only checkpoints that no downstream notebook consumes.
from publication_utils import remove_owned_intermediates

remove_owned_intermediates(BASE_DIR, [
    "adult_individual_h5ad_filtered",
    "adult_individual_nodoublet_h5ad",
    "adult_individual_h5ad/adata_adult_combined_filtered.h5ad"
])
print("Removed disposable preprocessing checkpoints.")
